# A.5 Declarations of model entities

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

Declarations of model entities have the following common form:

```ampl
entity name alias opt indexing opt body opt ;
```

where name is an alphanumeric name that has not previously been assigned to an entity by a declaration,
alias is an optional literal, indexing is an optional indexing expression, and entity is one of
the keywords

```ampl
set
param
var
arc
minimize
maximize
subject to
node
```

In addition, several other constructs are technically entity declarations, but are described later;
these include environ, problem, suffix and `table`.

The entity may be omitted, in which case `subject to` is assumed. The body of various declarations
consists of other, mostly optional, phrases that follow the initial part. Each declaration
ends with a semicolon.

Declarations may appear in any order, except that each name must be declared before it is used.

As with piecewise-linear terms, a special form of indexing expression is allowed for variable,
constraint, and objective declarations:

```ampl
{if lexpr}
```

If the logical expression lexpr is true, then a simple (unsubscripted) entity results; otherwise the
entity is excluded from the model, and subsequent attempts to reference it cause an error message.
For example, this declaration includes the variable Test in the model if the parameter testing
has been given a value greater than 100:

```ampl
param testing;
var Test {if testing > 100} >= 0;
```

A.6 Set declarations
A set declaration has the form

```ampl
set declaration:
       set name alias opt indexing opt attributes opt ;
```

in which attributes is a list of attributes optionally separated by commas:

```ampl
attribute:
      dimen n
      within sexpr
      = sexpr
      default sexpr
```

The dimension of the set is either the constant positive integer n, or the dimension of sexpr, or 1 by
default. The phrase within sexpr requires the set being declared to be a subset of sexpr. Several
within phrases may be given. The = phrase specifies a value for the set; it implies that the set
will not be given a value later in a data section (A.12.1) or a command such as let (A.18.9). The
default phrase specifies a default value for the set, to be used if no value is given in a data section.
The = and default phrases are mutually exclusive. If neither is given and the set is not
defined by a data statement, references to the set during model generation cause an error message.
For historical reasons, := is currently a synonym for = in declarations of sets and parameters, but
this use of := is deprecated.

The sexpr in a = or default phrase can be {}, the empty set, which then has the dimension
implied by any dimen or within phrases in the declaration, or 1 if none is present. In other
contexts, {} denotes the empty set.

Recursive definitions of indexed sets are allowed, so long as the assigned values can be computed
in a sequence that only references previously computed values. For example,

```ampl
set nodes;
set arcs within nodes cross nodes;
param max_iter = card(nodes)-1; # card(s) = number of elements in s
set step {s in 1..max_iter} dimen 2 =
    if s == 1
        then arcs
        else step[s-1] union setof {k in nodes,
               (i,k) in step[s-1], (k,j) in step[s-1]} (i,j);
set reach = step[max_iter];
```

computes in set reach the transitive closure of the graph represented by nodes and arcs.